# Understanding differential fairness by group pair

A scalar EDF value identifies the largest disparity but does not name the groups involved. This example uses synthetic weighted observations with two protected attributes. It reports the underlying group comparisons without changing the scalar metric. The data are invented for demonstration and require no download.

In [1]:
import numpy as np
import pandas as pd
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric

frame = pd.DataFrame({
    'region': [0, 0, 0, 0, 1, 1],
    'category': [0, 0, 1, 1, 0, 0],
    'selected': [0, 1, 0, 1, 0, 1],
    'weight': [4.5, 0.5, 2.5, 2.5, 0.5, 4.5],
})
dataset = BinaryLabelDataset(
    df=frame, label_names=['selected'],
    protected_attribute_names=['region', 'category'],
    instance_weights_name='weight')
metric = BinaryLabelDatasetMetric(dataset)

With concentration 1, the smoothed favorable-outcome probabilities are 1/6, 1/2, and 5/6. Comparing the first and third groups gives a probability ratio of 5 and hence an EDF value of log(5). Both favorable and unfavorable outcomes are considered.

In [2]:
details = metric.smoothed_empirical_differential_fairness_details(top_k=2)
assert np.isclose(details['epsilon'], np.log(5))
assert np.isclose(details['epsilon'], metric.smoothed_empirical_differential_fairness())
print('EDF:', round(details['epsilon'], 6))
for pair in details['pairs']:
    print(pair['groups'], 'epsilon:', round(pair['epsilon'], 6))

EDF: 1.609438
({'region': 0.0, 'category': 0.0}, {'region': 1.0, 'category': 0.0}) epsilon: 1.609438
({'region': 0.0, 'category': 1.0}, {'region': 1.0, 'category': 0.0}) epsilon: 1.098612


`top_k` limits the returned breakdown, while `maximizing_pairs` retains every pair attaining the global maximum. Only observed attribute combinations are compared. The method computes all pair comparisons, so reducing `top_k` does not reduce the pair-generation cost. Group differences describe these observations; they do not establish why a disparity occurred.

In [3]:
for pair in details['maximizing_pairs']:
    print('Maximum:', pair['groups'])
assert len(details['pairs']) == 2
assert len(details['maximizing_pairs']) == 1

Maximum: ({'region': 0.0, 'category': 0.0}, {'region': 1.0, 'category': 0.0})
